# 4교시. 멀티모달·생성형 AI 기반 핵심 정보 추출

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leecks1119/document_ai_lecture/blob/master/colab/04_genai_extraction.ipynb)

## 오늘 꼭 할 일

같은 영수증을 실제 PaddleOCR-VL로 읽고 Markdown·업무 JSON을 바로 확인합니다.

1. 제공 예제로 결과를 먼저 만듭니다.
2. 화면에서 이번 교시의 핵심 결과 한 가지를 확인합니다.
3. 시간이 남으면 다른 공개·비식별 자료로 반복하고 차이를 기록합니다.

**끝났다는 증거:** 화면의 `✅ 실습 완료`와
`course_outputs/paddleocr_vl_result.md` 파일

> Google Colab도 외부 클라우드입니다. 조직 승인 없는 개인·회사 문서는
> 업로드하지 않습니다. 필수 실습은 저장소의 비식별 공개·합성 샘플만
> 사용합니다.

필수 실습에서는 제공 예제를 사용합니다. 다른 자료를 사용한 경우에는
화면에 표시된 파일명이 내가 선택한 파일과 같은지 먼저 확인합니다.
이 교시는 제공 이미지를 실제 VLM으로 읽어야 완료됩니다. GPU가 없거나
모델 실행이 실패하면 성공 결과로 바꾸지 않습니다. T4 GPU를
선택하고 모델 셀부터 다시 실행합니다.


## 이 노트북에서 내가 하는 일

- **필수 실습:** 공개 영수증 한 장을 PaddleOCR-VL-1.6으로 직접 읽고 OCR+규칙 결과와 비교합니다.
- **내가 바꾸는 곳:** 이미지 한 장과 원본 대조가 필요한 필드 한 곳만 고릅니다.
- **인터넷 자료로 다시 실험:** 다른 공개·비식별 이미지로 실제 VLM을 다시 실행해 잘 읽은 구조와 빠진 필드를 기록합니다.

먼저 제공 예제로 끝까지 실행해 `✅ 실습 완료`를 확인하세요. 그다음
[공개·비식별 실습 자료 찾기](https://github.com/leecks1119/document_ai_lecture/blob/master/docs/public_practice_sources.md)를 보고
입력 한 장만 바꾸어 다시 실행합니다. 2교시에서 만든 결과 파일은
3~7교시에 이어 쓸 수 있습니다. 매 교시 마지막의 **다른 자료 실험
기록**에서 잘된 점과 실패한 점을 남깁니다.

> `🟢 그대로 실행하는 셀`은 수정하지 않습니다. `🟠 내가 짧게 바꾸는
> 셀`만 필수이고, `🔵 원하면 바꾸는 셀`은 시간이 남을 때 합니다.
> 정답은 모두 공개되어 있으므로 정답을 먼저 복사하고 결과를 관찰해도 됩니다.

## 코드 셀을 읽는 방법

각 코드 셀의 맨 위에는 `코드 읽기` 주석이 있습니다.

1. `수정하지 않습니다`라고 적힌 셀은 설명을 읽고 그대로 실행합니다.
2. 주황색 필수 `TODO`만 채웁니다. 파란색 선택 `TODO`는 건너뛰어도 됩니다.
3. 실행 출력에서 `코드 읽는 법`과 `확인할 결과`를 다시 확인합니다.
4. `단계 실행 완료`가 나온 뒤 다음 코드 셀로 이동합니다.
5. 길고 어려운 준비 코드는 접혀 있습니다. 제목 왼쪽의 화살표를 눌러
   펼칠 수 있지만, 처음에는 펼치지 않아도 됩니다.

Python 문법 전체를 먼저 이해할 필요는 없습니다. 변수에 어떤 값이 들어가고,
실행 뒤 어떤 결과가 달라지는지를 중심으로 읽습니다.


In [ ]:
#@title 🟢 0. 실습 환경 준비 — 그대로 실행 { display-mode: "form" }
def _show_learning_message(markdown_text):
    try:
        from IPython.display import Markdown, display
        display(Markdown(markdown_text))
    except ImportError:
        print(markdown_text)


def show_lab_step(
    current,
    total,
    title,
    action,
    expected,
    code_help,
    edit_kind,
):
    cell_kind = {
        "required": "🟠 내가 짧게 바꾸는 셀",
        "optional": "🔵 원하면 바꾸는 셀",
        "none": "🟢 그대로 실행하는 셀",
    }[edit_kind]
    _show_learning_message(
        f"""---
### {cell_kind} · {current}/{total} · {title}

**지금 할 일:** {action}

**코드 읽는 법:** {code_help}

**이 단계에서 확인할 결과:** {expected}
"""
    )


def complete_lab_step(current, total, expected):
    next_action = (
        "결과를 확인한 뒤 다음 코드 셀을 실행하세요."
        if current < total
        else "마지막 실습 완료 문구와 산출물 파일을 확인하세요."
    )
    _show_learning_message(
        f"""> ✅ **{current}/{total} 단계 실행 완료**
>
> **결과 확인:** {expected}
>
> **다음 행동:** {next_action}
"""
    )

# ── 코드 읽기 ─────────────────────────────────────────────
# `OUTPUT_DIR`와 `load_course_assets()`를 준비합니다. 수정하지 않습니다.
# ──────────────────────────────────────────────────────────
show_lab_step(1, 6, '공통 환경 준비', '결과 폴더와 자료 로더를 준비합니다.', '공통 작업 폴더가 표시되어야 합니다.', '`OUTPUT_DIR`와 `load_course_assets()`를 준비합니다. 수정하지 않습니다.', 'none')

import json
import os
import platform
import sys
from pathlib import Path

OUTPUT_DIR = Path("course_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
AUTOMATED_CHECK = os.getenv("COURSE_VALIDATE_EXAMPLE") == "1"

def upload_previous_artifact(filename):
    target = OUTPUT_DIR / filename
    if target.exists() or AUTOMATED_CHECK:
        return target if target.exists() else None
    try:
        from google.colab import files
    except ImportError:
        return None
    print(f"이전 교시에서 내려받은 {filename}을 선택하세요.")
    uploaded = files.upload()
    if filename not in uploaded:
        raise FileNotFoundError(
            f"{filename}이 선택되지 않았습니다. 자료 선택에서 "
            "'제공 예제'를 고르거나 파일을 다시 선택하세요."
        )
    target.write_bytes(uploaded[filename])
    return target


def download_artifact(path):
    if AUTOMATED_CHECK:
        return
    try:
        from google.colab import files
    except ImportError:
        return
    files.download(str(path))

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("공통 작업 폴더:", OUTPUT_DIR.resolve())

COURSE_ASSET_BASE_URL = (
    "https://raw.githubusercontent.com/leecks1119/"
    "document_ai_lecture/master/"
)

def load_course_assets(*relative_paths):
    if AUTOMATED_CHECK:
        local_root = os.getenv("COURSE_LOCAL_ASSET_ROOT")
        if not local_root:
            raise RuntimeError(
                "자동 검증용 COURSE_LOCAL_ASSET_ROOT가 필요합니다."
            )
        root = Path(local_root)
        return {
            path: (root / path).read_bytes()
            for path in relative_paths
        }

    import requests

    loaded = {}
    missing = []
    for path in relative_paths:
        try:
            response = requests.get(
                COURSE_ASSET_BASE_URL + path,
                timeout=30,
            )
            response.raise_for_status()
            loaded[path] = response.content
        except requests.RequestException as exc:
            print(f"자동 다운로드 실패: {Path(path).name} · {exc}")
            missing.append(path)

    if missing:
        from google.colab import files

        expected = ", ".join(Path(path).name for path in missing)
        print("다음 파일을 저장소에서 내려받아 선택하세요:", expected)
        uploaded = files.upload()
        uploaded_by_name = {
            Path(name).name: content
            for name, content in uploaded.items()
        }
        for path in missing:
            filename = Path(path).name
            if filename not in uploaded_by_name:
                raise FileNotFoundError(
                    f"{filename}이 선택되지 않았습니다."
                )
            loaded[path] = uploaded_by_name[filename]

    return loaded

complete_lab_step(1, 6, '공통 작업 폴더가 표시되어야 합니다.')


In [ ]:
#@title 🔵 실습 자료 고르기 { display-mode: "form" }
# ── 코드 읽기 ─────────────────────────────────────────────
# `INPUT_PATH`가 실제 VLM 입력이며 앞 교시 OCR 결과가 있으면 비교 기준으로 사용합니다.
# ──────────────────────────────────────────────────────────
show_lab_step(2, 6, 'VLM 입력 선택', '실제 VLM에 넣을 이미지 한 장을 고릅니다.', '원본 이미지·파일명·OCR 비교 기준을 확인합니다.', '`INPUT_PATH`가 실제 VLM 입력이며 앞 교시 OCR 결과가 있으면 비교 기준으로 사용합니다.', 'optional')

COURSE_PYTHON_PATHS = ['src/__init__.py', 'src/clean.py', 'src/export.py', 'src/extract.py', 'src/ocr.py', 'src/pipeline.py', 'src/sample_data.py', 'src/validate.py', 'src/vlm.py']
course_python_assets = load_course_assets(*COURSE_PYTHON_PATHS)
for relative_path, payload in course_python_assets.items():
    target = Path(relative_path)
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(payload)
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

# INPUT_FORM_CELL
import io
import requests
from PIL import Image
try:
    from IPython.display import display
except ImportError:
    display = print

SAMPLE_IMAGE_PATH = 'sample_docs/public_receipts/korea/taebaek_restaurant_2025_redacted.png'
sample_bytes = load_course_assets(SAMPLE_IMAGE_PATH)[SAMPLE_IMAGE_PATH]

# TODO(선택): 제공 예제 확인 뒤 입력 이미지만 바꾸어 다시 실행하세요.
실습_자료 = "제공 예제" #@param ["제공 예제", "내 컴퓨터에서 업로드", "인터넷 이미지 URL"]
인터넷_이미지_URL = "" #@param {type:"string"}
if AUTOMATED_CHECK:
    실습_자료 = "제공 예제"

input_bytes = sample_bytes
INPUT_FILE_NAME = Path(SAMPLE_IMAGE_PATH).name
if 실습_자료 == "내 컴퓨터에서 업로드":
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("이미지 한 장만 선택하세요.")
    INPUT_FILE_NAME, input_bytes = next(iter(uploaded.items()))
elif 실습_자료 == "인터넷 이미지 URL":
    if not 인터넷_이미지_URL.strip():
        raise ValueError("인터넷_이미지_URL에 이미지 주소를 입력하세요.")
    response = requests.get(인터넷_이미지_URL.strip(), timeout=30)
    response.raise_for_status()
    input_bytes = response.content
    INPUT_FILE_NAME = "internet_document.png"

if len(input_bytes) > 5 * 1024 * 1024:
    raise ValueError("수업에서는 5MB 이하 이미지 한 장만 처리합니다.")
input_image = Image.open(io.BytesIO(input_bytes)).convert("RGB")
INPUT_PATH = OUTPUT_DIR / 'vlm_input.png'
input_image.save(INPUT_PATH)
preview = input_image.copy()
preview.thumbnail((650, 750))
display(preview)
print("선택한 자료:", 실습_자료)
print("실제 모델 입력:", INPUT_FILE_NAME, input_image.size)

from src.extract import extract_receipt_from_text
from src.vlm import parse_with_paddleocr_vl, vlm_text_from_result

previous_path = OUTPUT_DIR / "clean_receipt.json"
if previous_path.exists():
    previous = json.loads(previous_path.read_text(encoding="utf-8"))
    OCR_BASELINE_TEXT = "\n".join(previous["cleaned_lines"])
    OCR_BASELINE_SOURCE = "3교시 OCR 행 묶기 결과"
else:
    OCR_BASELINE_TEXT = '이태리집\n거래일시 2025-10-04 12:33:37\n페퍼로니 앤 치즈 29,000 1 29,000\n토마토 파스타 14,000 1 14,000\n수제 돈가스 13,000 1 13,000\n새우 칠리치 필라 14,000 1 14,000\n콜라 2,000 3 6,000\n합계 금액 76,000\n부가세 과세물품가액 69,094\n부가세 6,906\n'
    OCR_BASELINE_SOURCE = "제공 영수증의 OCR 비교 원문"
print("비교 기준:", OCR_BASELINE_SOURCE)

complete_lab_step(2, 6, '원본 이미지·파일명·OCR 비교 기준을 확인합니다.')


In [ ]:
#@title 🟢 PaddleOCR-VL 실제 실행 — 그대로 실행 { display-mode: "form" }
# ── 코드 읽기 ─────────────────────────────────────────────
# `parse_with_paddleocr_vl()`이 `PaddleOCR-VL-1.6-0.9B`를 T4 GPU에서 실제 실행합니다.
# ──────────────────────────────────────────────────────────
show_lab_step(3, 6, 'PaddleOCR-VL 실제 실행', 'T4 GPU에서 현재 이미지를 실제 문서 VLM으로 읽습니다.', '모델 실행 여부와 실제 Markdown을 확인합니다.', '`parse_with_paddleocr_vl()`이 `PaddleOCR-VL-1.6-0.9B`를 T4 GPU에서 실제 실행합니다.', 'none')

import subprocess

VLM_PIPELINE_NAME = "PaddleOCR-VL-1.6"
VLM_MODEL_NAME = "PaddleOCR-VL-1.6-0.9B"
if not AUTOMATED_CHECK:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError(
            "T4 GPU가 필요합니다. 런타임 → 런타임 유형 변경에서 "
            "T4 GPU를 선택한 뒤 이 셀부터 다시 실행하세요."
        )
    print("GPU:", torch.cuda.get_device_name(0))
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "paddleocr[doc-parser]==3.7.0",
        "transformers>=5.8,<6",
    ])

if AUTOMATED_CHECK:
    vlm_result = {
        "model_executed": False,
        "pipeline": VLM_PIPELINE_NAME,
        "vlm_model": VLM_MODEL_NAME,
        "engine": "transformers",
        "input_file": INPUT_FILE_NAME,
        "pages": [],
        "automated_repository_check": True,
    }
else:
    vlm_result = parse_with_paddleocr_vl(
        INPUT_PATH,
        engine="transformers",
        device="gpu",
    )
    vlm_result["automated_repository_check"] = False

raw_path = OUTPUT_DIR / "paddleocr_vl_raw.json"
raw_path.write_text(
    json.dumps(vlm_result, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
vlm_markdown = vlm_text_from_result(vlm_result)
markdown_path = OUTPUT_DIR / "paddleocr_vl_result.md"
if vlm_result["model_executed"]:
    markdown_path.write_text(vlm_markdown + "\n", encoding="utf-8")

print("모델 실행 완료:", vlm_result["model_executed"])
print("실제 VLM:", VLM_MODEL_NAME)
if vlm_markdown:
    print("\n--- 모델이 복원한 Markdown 앞부분 ---")
    print(vlm_markdown[:1800])

complete_lab_step(3, 6, '모델 실행 여부와 실제 Markdown을 확인합니다.')


In [ ]:
#@title 🟢 업무 JSON 비교 — 그대로 실행 { display-mode: "form" }
# ── 코드 읽기 ─────────────────────────────────────────────
# `extract_receipt_from_text()`가 모델 Markdown을 업무 JSON으로 옮겨 비교합니다.
# ──────────────────────────────────────────────────────────
show_lab_step(4, 6, '업무 JSON 비교', 'OCR+규칙과 VLM 결과를 같은 필드로 비교합니다.', '총액·품목 수와 세 JSON 파일을 확인합니다.', '`extract_receipt_from_text()`가 모델 Markdown을 업무 JSON으로 옮겨 비교합니다.', 'none')

ocr_receipt = extract_receipt_from_text(
    OCR_BASELINE_TEXT,
    source_mode="OCR 원문 + 공개 Python 규칙",
)
vlm_receipt = extract_receipt_from_text(
    vlm_markdown,
    source_mode="PaddleOCR-VL 실제 결과 + 공개 Python 규칙",
)

receipt_path = OUTPUT_DIR / "receipt.json"
vlm_path = OUTPUT_DIR / "receipt_vlm.json"
comparison_path = OUTPUT_DIR / "vlm_comparison.json"
receipt_path.write_text(
    json.dumps(ocr_receipt, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
vlm_path.write_text(
    json.dumps(vlm_receipt, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
comparison = {
    "model_executed": vlm_result["model_executed"],
    "model": VLM_MODEL_NAME,
    "comparison": {
        field: {
            "ocr_rule": ocr_receipt.get(field),
            "actual_vlm": vlm_receipt.get(field),
        }
        for field in ("store_name", "date", "total_amount", "items")
    },
}
comparison_path.write_text(
    json.dumps(comparison, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
print("OCR+규칙 총액:", ocr_receipt["total_amount"])
print("실제 VLM 총액:", vlm_receipt["total_amount"])
print("실제 VLM 품목 수:", len(vlm_receipt["items"]))
print("✅ 실습 완료:", receipt_path, vlm_path, comparison_path)
download_artifact(vlm_path)

complete_lab_step(4, 6, '총액·품목 수와 세 JSON 파일을 확인합니다.')


In [ ]:
# ── 코드 읽기 ─────────────────────────────────────────────
# `반드시_확인할_필드` 한 곳만 바꿉니다. 모델 결과는 원본 대조 후 사용합니다.
# ──────────────────────────────────────────────────────────
show_lab_step(5, 6, '검토 필드 선택', '원본과 반드시 대조할 필드 하나를 정합니다.', '내 선택과 전체 정답 예시를 확인합니다.', '`반드시_확인할_필드` 한 곳만 바꿉니다. 모델 결과는 원본 대조 후 사용합니다.', 'required')

# TODO: 원본과 반드시 대조할 필드 하나를 적으세요.
반드시_확인할_필드 = "total_amount" #@param {type:"string"}
print("내 검토 대상:", 반드시_확인할_필드)
print("전체 정답 예시: total_amount · items · store_name은 원본 대조 후 사용")

complete_lab_step(5, 6, '내 선택과 전체 정답 예시를 확인합니다.')


## 선택 실험: 다른 자료로 한 번 더 확인하기

필수 실습을 먼저 끝낸 뒤, 인터넷에서 찾은 공개 문서나 개인정보를
가린 자료 한 장으로 같은 과정을 반복합니다. 결과가 잘 나오지 않아도
실패한 위치와 다음 질문을 남기면 실험이 완료됩니다.


In [ ]:
#@title 🔵 선택: 다른 자료 실험 기록 { display-mode: "form" }
# ── 코드 읽기 ─────────────────────────────────────────────
# 이 셀은 선택 실험 기록지입니다. 위쪽 입력칸만 채우면 자료 출처, 잘된 점, 실패한 점, 다음 질문을 Markdown 파일로 저장합니다.
# ──────────────────────────────────────────────────────────
show_lab_step(6, 6, '다른 자료 실험 기록', '인터넷에서 찾은 공개 자료나 비식별 자료의 결과를 네 줄로 정리합니다.', '`lesson04_research_note.md` 파일과 기록 내용이 표시되어야 합니다.', '이 셀은 선택 실험 기록지입니다. 위쪽 입력칸만 채우면 자료 출처, 잘된 점, 실패한 점, 다음 질문을 Markdown 파일로 저장합니다.', 'optional')

# RESEARCH_NOTE_CELL
# TODO(선택): 다른 자료로 다시 실험했다면 아래 입력칸만 채우세요.
자료_구분 = "제공 예제" #@param ["제공 예제", "공개 웹 자료", "비식별 개인 자료", "회사 승인 자료"]
자료_이름_또는_URL = "" #@param {type:"string"}
문서_종류 = "영수증" #@param ["영수증", "견적서", "신청서", "거래명세서", "표 캡처", "기타"]
잘된_점 = "" #@param {type:"string"}
실패한_점 = "" #@param {type:"string"}
다음_질문 = "" #@param {type:"string"}

research_focus = '추출된 필드와 빠진 필드, 원문 근거가 없는 값을 구분해 기록합니다.'
note = f'''# {문서_종류} 실험 기록

- 자료 구분: {자료_구분}
- 자료 이름 또는 원문 URL: {자료_이름_또는_URL or "미입력"}
- 이번 교시 관찰 질문: {research_focus}
- 잘된 점: {잘된_점 or "미입력"}
- 실패하거나 이상한 점: {실패한_점 or "미입력"}
- 다음에 바꿔 볼 한 가지: {다음_질문 or "미입력"}
'''
note_path = OUTPUT_DIR / "lesson04_research_note.md"
note_path.write_text(note + "\n", encoding="utf-8")
try:
    from IPython.display import Markdown, display
    display(Markdown(note))
except ImportError:
    print(note)
print("실험 기록 저장:", note_path)

complete_lab_step(6, 6, '`lesson04_research_note.md` 파일과 기록 내용이 표시되어야 합니다.')
